# Map, Reduce, and one rewrite rule, from scratch

**Task.** We get one long document: a news feed of 400 AG News articles concatenated together
(each is `[A001] Title. Description`). We want, for each category (World, Sports, Business, Sci/Tech),
one line: *"There are N articles. {one-line summary of their titles}"*.

## 0. Setup: pick one of two ways to run

**Option A: use your own OpenAI API key.** You can then change anything and run any prompt.
- In Colab: click the key icon on the left (*Secrets*), add `OPENAI_API_KEY` with your key as the value,
  and turn on notebook access. Running locally: put `OPENAI_API_KEY=sk-...` in a `.env` file.
- Running the whole notebook from scratch with `gpt-5-nano` costs about **$1.14**.
- Every answer is saved (cached), so sending the exact same request again costs nothing.

**Option B: no key.** Just skip the key.
- We already ran this notebook and saved every LLM answer. Run it as is, and it uses those saved answers:
  free, fast, and no key needed.
- If you change what is sent to the LLM (a prompt, the model, the chunk size), there is no saved answer for it,
  so that cell stops with an error. To try such changes, switch to option A.

In [ ]:
# Get the lab files. In Colab: download the data + shared LLM cache and install packages. Locally: nothing to do.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ruiying-Ma/berkeley-cs294-316-lab.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("data/agnews_feed.json").exists():
    if not Path("berkeley-cs294-316-lab").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "berkeley-cs294-316-lab"], check=True)
    os.chdir("berkeley-cs294-316-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "jinja2", "python-dotenv"], check=True)

# Optional: your own OpenAI API key, stored in Colab's Secrets panel (key icon) as OPENAI_API_KEY.
if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass  # no key: the notebook runs in cache-only mode

### An LLM call that returns a JSON

In [ ]:
# pip install openai jinja2 python-dotenv matplotlib
import json, os, hashlib, random, textwrap, threading, time
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

from jinja2 import Template
from openai import OpenAI, RateLimitError
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY (and optionally OPENAI_BASE_URL) from .env
MODEL = "gpt-5-nano"
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY, base_url=os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE")) if API_KEY else None
print("API key found: prompts that are not in the cache will be sent to the LLM (and paid for)." if client else
      "No API key: cache-only mode. Prompts in the cache are free; any other prompt raises an error.")

CACHE = Path(".llm_cache"); CACHE.mkdir(exist_ok=True)
STATS = {"llm_calls": 0, "input_tokens": 0, "output_tokens": 0}  # counted even for cached answers
_lock = threading.Lock()

MAX_API_CALLS = 415  # safety cap on real (non-cached) LLM requests made by this notebook
API_CALLS = 0

WIDTH = 100  # max characters per printed line, so outputs don't need horizontal scrolling

def wprint(text=""):
    """print(), but wrap every line longer than WIDTH (continuation lines are indented)."""
    for line in str(text).split("\n"):
        lead = line[: len(line) - len(line.lstrip())]
        print(textwrap.fill(line, WIDTH, initial_indent="", subsequent_indent=lead + "    ") if line.strip() else line)

def build_prompt(prompt: str, output_schema: dict) -> str:
    """The exact text sent to the LLM: the prompt plus instructions for the JSON answer."""
    schema = "\n".join(f'  "{k}": {v}' for k, v in output_schema.items())
    return f"{prompt}\n\nRespond with a JSON object with exactly these keys:\n{{\n{schema}\n}}"

def call_llm(prompt: str, output_schema: dict) -> dict:
    """Send a prompt, get back a dict with the keys of output_schema. Cached on disk."""
    prompt = build_prompt(prompt, output_schema)
    cache_file = CACHE / (hashlib.sha256((MODEL + prompt).encode()).hexdigest() + ".json")
    cached = json.loads(cache_file.read_text()) if cache_file.exists() else None
    if cached and "result" in cached:
        result, usage = cached["result"], cached["usage"]
    else:
        if client is None:
            raise RuntimeError("This prompt is not in the cache, and no API key is set (cache-only mode). "
                               "Add your OPENAI_API_KEY to run new prompts (see the Setup section).")
        global API_CALLS
        with _lock:
            if API_CALLS >= MAX_API_CALLS:
                raise RuntimeError(f"LLM call cap reached: {MAX_API_CALLS:,} API calls made. "
                                   "Raise MAX_API_CALLS if you really want more.")
            API_CALLS += 1
        for attempt in range(8):
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"},
                )
                break
            except RateLimitError:
                time.sleep(random.uniform(1, 2 ** attempt))  # rate limited: wait a random while, retry
        answer = json.loads(response.choices[0].message.content)
        result = {k: answer.get(k) for k in output_schema}
        usage = {"input_tokens": response.usage.prompt_tokens, "output_tokens": response.usage.completion_tokens}
        cache_file.write_text(json.dumps({"result": result, "usage": usage}))
    with _lock:
        STATS["llm_calls"] += 1
        STATS["input_tokens"] += usage["input_tokens"]
        STATS["output_tokens"] += usage["output_tokens"]
    return result


# Quick check that the LLM call works (should print {"answer": "Paris"})
print(call_llm("What is the capital of France?", {"answer": "string"}))

### Task Input: One document of 400 news articles

In [ ]:
docs = json.loads(Path("data/agnews_feed.json").read_text())
print(len(docs), "document,", len(docs[0]["content"]), "characters (~", len(docs[0]["content"]) // 4, "tokens)")
wprint(docs[0]["content"][:600] + " ...\n")
wprint("Per-category distribution:")
GROUND_TRUTH = Path("data/agnews_ground_truth.txt").read_text().split("Article id")[0]
print(GROUND_TRUTH)

## 1. The map operator

**map(docs, prompt, output_schema)**: for each document, fill the prompt template with that document
(`{{ input.<field> }}`), ask the LLM, and add the LLM's answer to the document.

In [ ]:
def map_op(docs, prompt, output_schema):
    def process(doc):
        rendered = Template(prompt).render(input=doc)
        return {**doc, **call_llm(rendered, output_schema)}
    with ThreadPoolExecutor(max_workers=8) as pool:   # = [process(d) for d in docs], in parallel
        return list(pool.map(process, docs))

The map for our task: go through the feed and give, for every article, its id, title and category.

In [ ]:
MAP_PROMPT = """Here is a news feed. Each article starts with an id like [A001], then its title, then its content.

{{ input.content }}

For every article in the feed, give its id (e.g. "A001"), its title, and its category,
which must be exactly one of: World, Sports, Business, Sci/Tech. Do not skip any article."""

MAP_SCHEMA = {"articles": "list[{id: string, title: string, category: World|Sports|Business|Sci/Tech}]"}

## 2. The reduce operator

**reduce(docs, reduce_key, prompt, output_schema)**: group the documents that share the same value of
`reduce_key`, then make one LLM call per group. The prompt sees the whole group as `inputs`.
Many documents in, one document per group out.

In [ ]:
def reduce_op(docs, reduce_key, prompt, output_schema):
    groups = {}
    for doc in docs:
        groups.setdefault(doc[reduce_key], []).append(doc)

    def process(item):
        key, group = item
        rendered = Template(prompt).render(inputs=group, reduce_key=key)
        return {reduce_key: key, **call_llm(rendered, output_schema)}
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(process, groups.items()))

The reduce for our task: one call per category, summarizing from the article titles. The count is the group size (`inputs | length`), so the LLM does not have to count.

In [ ]:
REDUCE_PROMPT = """Below are titles of news articles:
{% for input in inputs %}- {{ input.title }}
{% endfor %}
Based on these titles, write one line of the form "There are {{ inputs | length }} articles. <one-line summary of what these articles are about>"."""

REDUCE_SCHEMA = {"summary": "string"}

## 3. Other operators

- **unnest(docs, key)**: the map gives one document holding a list of articles. Unnest turns
  `{articles: [{id, title, category}, ...]}` into one document per article: `{id, title, category}`.
- **split(docs, split_key, articles_per_chunk)**: the opposite direction: cut one long document into
  several smaller ones. We cut at article boundaries (articles are separated by a blank line).
- ...

In [ ]:
def unnest(docs, key):
    """One document per item of doc[key] (dict items are merged into the document)."""
    out = []
    for doc in docs:
        rest = {k: v for k, v in doc.items() if k != key}
        for item in doc.get(key) or []:
            out.append({**rest, **item} if isinstance(item, dict) else {**rest, key: item})
    return out


def split_op(docs, split_key, articles_per_chunk):
    """Cut doc[split_key] at article boundaries; each chunk of articles becomes its own document."""
    out = []
    for doc_id, doc in enumerate(docs):
        articles = doc[split_key].split("\n\n")
        for i in range(0, len(articles), articles_per_chunk):
            chunk = "\n\n".join(articles[i:i + articles_per_chunk])
            out.append({**doc, split_key: chunk, "doc_id": doc_id, "chunk_id": i // articles_per_chunk})
    return out

## 4. A naive pipeline: map → unnest → reduce

A pipeline is a list of operators.

In [ ]:
OPERATORS = {"map": map_op, "reduce": reduce_op, "unnest": unnest, "split": split_op}

def now():
    return datetime.now().strftime("%H:%M:%S")

LOG_INDENT = " " * 11  # lines up with the text after "[hh:mm:ss] "

def log_docs(docs):
    """Print a compact preview of docs, indented under the log line."""
    for line in preview(docs, max_items=2, max_chars=25, width=WIDTH - len(LOG_INDENT)).split("\n"):
        print(LOG_INDENT + line)


def example_prompt(step, docs):
    """The rendered prompt of the step's first LLM call (first doc for map, first group for reduce)."""
    if step["type"] == "map" and docs:
        rendered = Template(step["prompt"]).render(input=docs[0])
        n_calls = len(docs)
    elif step["type"] == "reduce" and docs:
        key = docs[0][step["reduce_key"]]
        group = [d for d in docs if d[step["reduce_key"]] == key]
        rendered = Template(step["prompt"]).render(inputs=group, reduce_key=key)
        n_calls = len({d[step["reduce_key"]] for d in docs})
    else:
        return None, 0
    return build_prompt(rendered, step["output_schema"]), n_calls


def log_prompt(step, docs, head=6, tail=6):
    """Print the first rendered prompt of a step, wrapped; long prompts keep only head and tail lines."""
    prompt, n_calls = example_prompt(step, docs)
    if prompt is None:
        return
    body_width = WIDTH - len(LOG_INDENT) - 2
    lines = [w for line in prompt.split("\n") if line.strip() for w in textwrap.wrap(line, body_width)]
    if len(lines) > head + tail:
        lines = lines[:head] + [f"… {len(lines) - head - tail} more lines …"] + lines[-tail:]
    print(f"{LOG_INDENT}prompt of LLM call 1 of {n_calls}:")
    for line in lines:
        print(f"{LOG_INDENT}│ {line}")


def run_pipeline(pipeline, docs):
    """Run the steps in order, printing a live log; returns a trace of every step's output."""
    trace = {"input": docs, "steps": []}
    print(f"[{now()}] INPUT  {len(docs)} docs")
    log_docs(docs)
    for i, step in enumerate(pipeline, 1):
        args = {k: v for k, v in step.items() if k not in ("type", "name")}
        params = {k: v for k, v in args.items() if k not in ("prompt", "output_schema")}
        label = f"{step['type']}({step['name']})"
        wprint(f"\n[{now()}] ▶ step {i}/{len(pipeline)}: {label} {params if params else ''}")
        log_prompt(step, docs)

        before, start = dict(STATS), time.time()
        n_in = len(docs)
        docs = OPERATORS[step["type"]](docs, **args)
        used = {k: STATS[k] - before[k] for k in STATS}
        trace["steps"].append({"step": step, "output": docs, **used})

        print(f"[{now()}] ✓ {label}  ({time.time() - start:.1f}s)")
        print(f"{LOG_INDENT}{n_in} → {len(docs)} docs | {used['llm_calls']} LLM calls | "
              f"{used['input_tokens']:,} in / {used['output_tokens']:,} out tokens")
        log_docs(docs)

    total = {k: sum(s[k] for s in trace["steps"]) for k in STATS}
    print(f"\n[{now()}] DONE  {total['llm_calls']} LLM calls | "
          f"{total['input_tokens']:,} in / {total['output_tokens']:,} out tokens")
    return trace


def preview(x, indent=0, used=0, max_items=3, max_chars=40, width=WIDTH):
    """Show the structure of x, shortening long strings and long lists.

    Short dicts/lists stay on one line; if that line would pass `width`, put one entry per line.
    `used` = characters already on the current line before x.
    """
    pad = " " * indent
    if isinstance(x, str):
        return json.dumps(x if len(x) <= max_chars else x[:max_chars] + "…", ensure_ascii=False)
    if isinstance(x, dict):
        keys = [f'"{k}": ' for k in x]
        values = [preview(v, indent + 2, indent + 2 + len(k), max_items, max_chars, width) for k, v in zip(keys, x.values())]
        parts, (open_, close) = [k + v for k, v in zip(keys, values)], "{}"
    elif isinstance(x, list):
        values = [preview(v, indent + 2, indent + 2, max_items, max_chars, width) for v in x[:max_items]]
        parts, (open_, close) = values + ([f"… +{len(x) - max_items} more"] if len(x) > max_items else []), "[]"
    else:
        return json.dumps(x)
    one_line = open_ + ", ".join(parts) + close
    if "\n" not in one_line and used + len(one_line) + 1 <= width:  # +1 for a trailing comma
        return one_line
    return open_ + "\n" + ",\n".join(pad + "  " + p for p in parts) + "\n" + pad + close


def show_pipeline(pipeline):
    wprint("  ->  ".join(f"{s['type']}({s['name']})" for s in pipeline))


def show_final(trace):
    """The final per-category lines, in full."""
    for r in sorted(trace["steps"][-1]["output"], key=lambda r: r["category"]):
        wprint(f"[{r['category']}] {r['summary']}")

naive_pipeline = [
    {"type": "map",    "name": "categorize_articles", "prompt": MAP_PROMPT, "output_schema": MAP_SCHEMA},
    {"type": "unnest", "name": "one_doc_per_article", "key": "articles"},
    {"type": "reduce", "name": "summarize_category", "reduce_key": "category",
     "prompt": REDUCE_PROMPT, "output_schema": REDUCE_SCHEMA},
]
show_pipeline(naive_pipeline)
print()
trace = run_pipeline(naive_pipeline, docs)
print()
show_final(trace)

Compare with the ground truth (World 160, Sports 120, Business 80, Sci/Tech 40, 400 total).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

CATEGORIES = ["World", "Sports", "Business", "Sci/Tech"]
LETTER = {"World": "W", "Sports": "S", "Business": "B", "Sci/Tech": "T"}
TRUTH = dict(line.split("\t") for line in  # article id -> true category
             Path("data/agnews_ground_truth.txt").read_text().split("Article id -> category:\n")[1].splitlines() if line)


def map_predictions(trace):
    """{article id: category} as returned by the map step of a pipeline run."""
    step = next(s for s in trace["steps"] if s["step"]["type"] == "map")
    predicted = {}
    for doc in step["output"]:
        for article in doc.get("articles") or []:
            aid = str(article.get("id", "")).strip("[] ").upper()
            if aid in TRUTH and aid not in predicted:  # ignore made-up and repeated ids
                predicted[aid] = article.get("category")
    return step["step"]["name"], predicted


def plot_map_accuracy(trace, per_row=40):
    """One figure: every article's outcome in feed order (left) and accuracy per category (right)."""
    name, predicted = map_predictions(trace)
    ids = list(TRUTH)
    outcome = ["missing" if a not in predicted else "correct" if predicted[a] == TRUTH[a] else "wrong" for a in ids]
    colors = {"correct": "#4caf50", "wrong": "#e53935", "missing": "#d9d9d9"}

    fig, (grid, bars) = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [3.3, 1]})

    # Left: one square per article, in feed order; letter = its true category
    for i, (aid, out) in enumerate(zip(ids, outcome)):
        row, col = divmod(i, per_row)
        grid.add_patch(plt.Rectangle((col, row), 0.92, 0.92, color=colors[out]))
        grid.text(col + 0.46, row + 0.5, LETTER[TRUTH[aid]], ha="center", va="center", fontsize=6,
                  color="white" if out != "missing" else "#666666")
    n_rows = -(-len(ids) // per_row)
    grid.set_xlim(0, per_row); grid.set_ylim(n_rows, 0); grid.set_aspect("equal")
    grid.set_xticks([]); grid.set_yticks([r + 0.46 for r in range(n_rows)])
    grid.set_yticklabels([ids[r * per_row] for r in range(n_rows)], fontsize=8)
    grid.tick_params(length=0)
    for side in grid.spines.values():
        side.set_visible(False)
    grid.set_title("Every article in feed order  (letter = true category: W World, S Sports, B Business, T Sci/Tech)",
                   fontsize=9, loc="left")
    grid.legend(handles=[Patch(color=c, label=k) for k, c in colors.items()],
                loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False, fontsize=9)

    # Right: per true category, how many articles were correct / wrong / missing
    for y, cat in enumerate(CATEGORIES):
        cat_out = [o for a, o in zip(ids, outcome) if TRUTH[a] == cat]
        left = 0
        for kind in ("correct", "wrong", "missing"):
            k = cat_out.count(kind)
            bars.barh(y, k, left=left, color=colors[kind])
            if k:
                bars.text(left + k / 2, y, str(k), ha="center", va="center", fontsize=8)
            left += k
        bars.text(left + 3, y, f"{cat_out.count('correct') / len(cat_out):.0%}", va="center", fontsize=9)
    bars.set_yticks(range(len(CATEGORIES))); bars.set_yticklabels(CATEGORIES)
    bars.invert_yaxis()
    bars.set_xlim(0, max(list(TRUTH.values()).count(c) for c in CATEGORIES) * 1.25)
    bars.set_xlabel("articles")
    bars.set_title("Accuracy per true category", fontsize=9, loc="left")
    for side in ("top", "right"):
        bars.spines[side].set_visible(False)

    correct = outcome.count("correct")
    fig.suptitle(f"map({name}) vs. ground truth:  {len(predicted)}/{len(ids)} articles returned,  "
                 f"{correct} correct  ({correct / len(ids):.0%} of all articles)", fontsize=11, x=0.01, ha="left")
    fig.tight_layout()
    plt.show()


plot_map_accuracy(trace)

## 5. The rewrite directive (map ⇒ split → map) and the optimized pipeline

One LLM call over 400 articles is too much to do carefully. The rewrite: **cut the document into
small chunks first, and run the very same map on each chunk.** The prompt does not change.

We don't need anything new to put the pieces back together: the unnest + reduce by category that
come next already gather each category's articles from all chunks.

In [ ]:
def rewrite_map_to_split_map(pipeline, map_name, articles_per_chunk):
    """Rewrite directive: replace the map step `map_name` with split -> map."""
    new_pipeline = []
    for step in pipeline:
        if step["type"] == "map" and step["name"] == map_name:
            field = "content"   # the field the map reads, {{ input.content }}
            new_pipeline.append({"type": "split", "name": f"split_{field}",
                                 "split_key": field, "articles_per_chunk": articles_per_chunk})
            new_pipeline.append({**step, "name": f"{step['name']}_per_chunk"})  # same prompt, same schema
        else:
            new_pipeline.append(step)
    return new_pipeline

In [ ]:
optimized_pipeline = rewrite_map_to_split_map(naive_pipeline, "categorize_articles", articles_per_chunk=1)

print("before:"); show_pipeline(naive_pipeline)
print("-----")
print("after: "); show_pipeline(optimized_pipeline)

Run the optimized pipeline: same map prompt, same reduce prompt, but each map call now sees one article instead of 400.

In [ ]:
optimized_trace = run_pipeline(optimized_pipeline, docs)
print()
show_final(optimized_trace)

In [ ]:
plot_map_accuracy(optimized_trace)